In [ ]:
from collections import defaultdict   # словарь, который для нового ключа сам создаёт значение по умолчанию (у нас пустой список)
from itertools import combinations   # перебирает все наборы из k элементов списка; нужен эталонному полному перебору в тестах
import random    # генерация случайных графов для тестов
import heapq    # куча: всегда отдаёт минимум за O(log n); нужна Приму, которым перепроверяем Краскала


# Граф

def build_graph(edges, vertices=()):  # строим граф из списка рёбер (u, v, w); vertices — вершины, у которых может не быть рёбер
    g = defaultdict(list)    # граф = словарь «вершина → список пар (сосед, вес)»; новая вершина сразу получает пустой список
    for v in vertices:   # изолированные вершины рёбрами не создадутся, поэтому заводим их отдельно
        g[v]    # само обращение к defaultdict создаёт ключ v с пустым списком, присваивать ничего не нужно
    for u, v, w in edges:    # перебираем рёбра, сразу распаковывая тройку на два конца и вес
        g[u].append((v, w))   # записываем, что из u можно пройти в v, и это стоит w
        g[v].append((u, w))    # и обратно из v в u: граф неориентированный, поэтому ребро хранится у обоих концов
    return g        # готовый словарь смежности


def graph_edges(graph):    # достаём из графа список рёбер, где каждое ребро встречается ровно один раз
    order = {v: i for i, v in enumerate(graph)}  # нумеруем вершины по порядку в словаре; номера нужны для выбора одной записи из двух
    edges = []     # сюда складываем рёбра тройками (u, v, w)
    for u in graph:     # идём по всем вершинам графа
        for v, w in graph[u]:  # и по всем её соседям — то есть по всем записям «из u в v с весом w»
            if order[u] < order[v]:   # ребро A–B записано дважды (A→B и B→A), берём только запись от меньшего номера к большему; петли u→u отсеются сами
                edges.append((u, v, w))  # запись прошла проверку — сохраняем ребро
    return edges     # список всех рёбер без повторов


# Система непересекающихся множеств

class DSU:   # Union-Find: хранит разбиение вершин на компоненты, каждая компонента — дерево с корнем
    def __init__(self, vertices):     # создаём структуру по списку вершин
        self.parent = {v: v for v in vertices} # вначале каждая вершина — отдельная компонента и сама себе родитель, то есть корень
        self.rank = {v: 0 for v in vertices}  # ранг — примерная высота дерева компоненты; у дерева из одной вершины он 0
        self.components = len(self.parent)   # сколько сейчас компонент: сначала столько же, сколько вершин, каждое слияние уменьшает на 1

    def find(self, v):   # возвращает корень компоненты v; две вершины связаны, если корень у них общий
        root = v    # начинаем подниматься вверх от самой вершины v
        while self.parent[root] != root:  # корень — единственная вершина, которая сама себе родитель; пока не дошли до неё
            root = self.parent[root]  # шагаем к родителю, то есть на этаж выше
        while self.parent[v] != root: # второй проход по тому же пути — сжатие путей: перевешиваем каждую вершину пути прямо на корень
            self.parent[v], v = root, self.parent[v]  # правая часть считается первой: вешаем v на корень и переходим к её старому родителю
        return root   # корень найден; цикл вместо рекурсии, чтобы не упасть с RecursionError на длинных цепочках

    def union(self, a, b):   # объединяет компоненты a и b; True — объединили, False — они и так были вместе
        ra, rb = self.find(a), self.find(b)  # сравнивать нужно корни компонент, а не сами вершины
        if ra == rb:    # корень общий — a и b уже связаны каким-то путём
            return False              # ребро a–b вместе с этим путём замкнуло бы цикл, сообщаем Краскалу «не брать»
        if self.rank[ra] < self.rank[rb]:  # хотим, чтобы ra было более высоким деревом
            ra, rb = rb, ra    # если это не так, меняем их местами
        self.parent[rb] = ra   # подвешиваем низкое дерево под высокое — так общая высота не растёт и find остаётся быстрым
        if self.rank[ra] == self.rank[rb]:  # высота вырастает только если деревья были одинаковой высоты:
            self.rank[ra] += 1   # одно встало под другое и добавило этаж
        self.components -= 1   # было две компоненты, стала одна
        return True   # объединение произошло — для Краскала это значит, что ребро цикла не создаёт


#  Краскал

def kruskal_mst(graph):  # минимальный остов; возвращает (список рёбер остова, суммарный вес); на несвязном графе — лес
    dsu = DSU(graph.keys())   # DSU по всем вершинам: вначале ни одно ребро не взято, каждая вершина сама по себе
    tree, total = [], 0    # рёбра, попавшие в остов, и их общий вес
    n = len(graph)    # число вершин; в остовном дереве всегда ровно n - 1 ребро
    for u, v, w in sorted(graph_edges(graph), key=lambda e: e[2]):  # идём по рёбрам от лёгких к тяжёлым; key говорит сортировать по третьему элементу, весу; это O(E log E)
        if dsu.union(u, v):  # union сразу и проверяет, и склеивает: True — концы были в разных компонентах, цикла нет
            tree.append((u, v, w))  # жадно берём ребро: оно самое лёгкое из оставшихся и цикл не создаёт (свойство разреза)
            total += w     # прибавляем его вес к весу остова
            if len(tree) == n - 1:  # набрали n - 1 ребро — все вершины уже связаны, любое следующее ребро дало бы цикл
                break    # поэтому досрочно выходим, на правильность это не влияет
    return tree, total   # на несвязном графе n - 1 рёбер не наберётся, и вернётся лес: по дереву на компоненту


#  Проверка корректности остова

def count_components(graph):  # сколько в графе связных кусков
    dsu = DSU(graph.keys())   # новый DSU, все вершины пока порознь
    for u, v, _ in graph_edges(graph):  # перебираем рёбра графа; вес не нужен, поэтому переменная _
        dsu.union(u, v)   # склеиваем концы — в итоге связанные вершины окажутся в одной компоненте
    return dsu.components   # сколько компонент осталось, столько кусков у графа


def has_cycle(vertices, edges):  # есть ли цикл среди данных рёбер
    dsu = DSU(vertices)   # DSU по вершинам
    return any(not dsu.union(u, v) for u, v, _ in edges)  # union вернул False — концы уже были связаны, ребро замкнуло цикл; any остановится на первом таком


def is_spanning_forest(graph, tree):  # проверяет, что tree — правильный остов (лес) графа, не глядя на минимальность веса
    real = {(u, v, w) for u in graph for v, w in graph[u]}  # множество всех рёбер графа в обоих направлениях, чтобы порядок концов был не важен
    if any((u, v, w) not in real for u, v, w in tree):  # в остове есть ребро, которого нет в графе (или с другим весом) —
        return False                  # значит остов неправильный
    if has_cycle(graph.keys(), tree): # в дереве циклов быть не может
        return False                  # нашёлся цикл — не дерево
    return len(tree) == len(graph) - count_components(graph)  # без циклов рёбер ровно V - k тогда и только тогда, когда всё связанное в графе связано и в остове


def brute_force_mst_weight(graph):  # эталон: вес минимального остова полным перебором (только для маленьких связных графов)
    edges, n = graph_edges(graph), len(graph)  # все рёбра и число вершин
    best = None     # лучший найденный вес; None — пока ни одного остова не нашли
    for subset in combinations(edges, n - 1):  # перебираем каждый набор из n - 1 рёбер — кандидат в остов
        if not has_cycle(graph.keys(), subset):  # n - 1 ребро без цикла на n вершинах — это обязательно остовное дерево
            w = sum(e[2] for e in subset)  # вес этого остова — сумма третьих элементов троек
            best = w if best is None else min(best, w)  # первый найденный запоминаем, дальше оставляем меньший
    return best     # заведомо правильный минимум; работает экспоненциально, поэтому только в тестах


def prim_weight(graph, start):   # алгоритм Прима: второй, независимый способ получить вес остова для сверки
    used, total = {start}, 0     # дерево растёт из стартовой вершины; used — вершины, уже попавшие в дерево
    heap = [(w, v) for v, w in graph[start]]  # рёбра из старта парами (вес, вершина): вес первым, чтобы куча упорядочивала по весу
    heapq.heapify(heap)    # превращаем список в кучу, теперь самое лёгкое ребро всегда в heap[0]
    while heap and len(used) < len(graph):  # пока есть рёбра-кандидаты и в дереве ещё не все вершины
        w, v = heapq.heappop(heap)   # достаём самое лёгкое ребро из дерева наружу
        if v in used:    # вершина успела попасть в дерево — ребро ведёт внутрь и дало бы цикл
            continue   # такую устаревшую запись просто пропускаем
        used.add(v)    # присоединяем вершину к дереву
        total += w     # и прибавляем вес ребра, по которому она пришла
        for u, wu in graph[v]: # у дерева появились новые рёбра наружу — те, что выходят из v
            if u not in used:  # рёбра внутрь дерева в кучу не кладём, они всё равно бесполезны
                heapq.heappush(heap, (wu, u))  # добавляем ребро-кандидат в кучу
    return total                      # вес остова по Приму, он обязан совпасть с Краскалом


#  Тесты

WEDGES = [('A', 'B', 4), ('A', 'C', 2), ('B', 'C', 1), ('B', 'D', 5),   # граф с занятия 9 ноября: 6 вершин, 9 рёбер,
          ('C', 'D', 8), ('C', 'E', 10), ('D', 'E', 2), ('D', 'F', 6),  # минимальный остов у него весит 13
          ('E', 'F', 3)]


def test_dsu():  # проверяем саму структуру Union-Find
    d = DSU([1, 2, 3, 4, 5])   # пять вершин, пока пять отдельных компонент
    assert d.components == 5   # счётчик компонент верный с самого начала
    assert d.union(1, 2) is True  # 1 и 2 были в разных компонентах — склеились, вернулся True
    assert d.union(3, 4) is True # так же склеиваем 3 и 4
    assert d.union(1, 2) is False # второй раз нельзя: они уже вместе — это ровно та ситуация, где ребро дало бы цикл
    assert d.find(1) == d.find(2)  # у склеенных вершин общий корень
    assert d.find(1) != d.find(3)  # а компоненты {1, 2} и {3, 4} друг с другом пока не связаны
    assert d.components == 3  # сейчас компоненты такие: {1, 2}, {3, 4}, {5}
    d.union(2, 3)     # сливаем две пары в одну компоненту {1, 2, 3, 4}
    d.find(4)    # вызов find по дороге должен сжать путь от 4 до корня
    assert d.parent[4] == d.find(4)   # проверяем сжатие: родитель вершины 4 теперь сразу корень
    big = DSU(range(100_000))  # длинная цепочка из 100 000 вершин — на ней рекурсивный find упал бы
    for i in range(99_999):   # соединяем 0-1, 1-2, 2-3, ... в одну цепь
        big.union(i, i + 1)
    assert big.components == 1   # вся цепочка стала одной компонентой, и ничего не упало


def test_basic():   # основной граф с занятия
    g = build_graph(WEDGES)     # строим граф
    tree, total = kruskal_mst(g)   # запускаем Краскала
    assert total == 13     # вес совпал с ответом с занятия
    assert len(tree) == 5     # 6 вершин → в остове 6 - 1 = 5 рёбер
    assert sum(w for _, _, w in tree) == total  # возвращённый вес совпадает с суммой весов самих рёбер
    assert {frozenset(e[:2]) for e in tree} == {  # frozenset убирает порядок концов: ребро A–C и C–A — одно и то же
        frozenset(p) for p in [('B', 'C'), ('A', 'C'), ('D', 'E'),  # ожидаемые рёбра остова
                               ('E', 'F'), ('B', 'D')]}
    assert is_spanning_forest(g, tree)  # и структурно это настоящий остов: рёбра из графа, без циклов, всё связано


def test_edge_sorting():     # проверяем, что рёбра действительно берутся по возрастанию веса
    g = build_graph(WEDGES)
    tree, _ = kruskal_mst(g)    # вес здесь не нужен, поэтому _
    weights = [w for _, _, w in tree] # веса рёбер в том порядке, в котором Краскал их добавлял
    assert weights == sorted(weights) # порядок неубывающий — значит, рёбра шли отсортированными
    assert not any(frozenset(e[:2]) == {'A', 'B'} for e in tree)  # ребро A–B (4) лёгкое, но A и B уже связаны через C, оно замкнуло бы цикл


def test_disconnected():     # несвязный граф: остовного дерева нет, должен получиться лес
    g = build_graph([(0, 1, 5), (1, 2, 1), (0, 2, 3), (7, 8, 2)],  # треугольник 0-1-2 и отрезок 7-8...
                    vertices=['Z'])   # ...плюс вершина Z вообще без рёбер — всего три куска
    tree, total = kruskal_mst(g)
    assert count_components(g) == 3   # убеждаемся, что кусков действительно три
    assert len(tree) == 6 - 3   # в лесу V - (число компонент) рёбер: 6 - 3 = 3
    assert total == 1 + 3 + 2    # из треугольника взяты 1 и 3 (ребро 5 дало бы цикл), из отрезка — его ребро 2
    assert is_spanning_forest(g, tree)  # лес структурно корректен
    assert kruskal_mst(build_graph([], vertices=[1, 2, 3])) == ([], 0)  # три вершины без рёбер: брать нечего, вес 0


def test_equal_weights():             # одинаковые веса: остов не единственный, проверяем вес и структуру, а не конкретные рёбра
    g = build_graph([(1, 2, 1), (2, 3, 1), (1, 3, 1)])  # треугольник из рёбер веса 1: годится любой из трёх остовов
    tree, total = kruskal_mst(g)
    assert total == 2 and len(tree) == 2  # любой остов здесь — два ребра общим весом 2
    assert not has_cycle(g.keys(), tree)  # третье ребро, замыкающее треугольник, не взято
    k5 = build_graph([(a, b, 7) for a, b in combinations(range(5), 2)])  # полный граф на 5 вершинах: все 10 пар соединены, все веса 7
    tree, total = kruskal_mst(k5)
    assert total == 28 and is_spanning_forest(k5, tree)  # любой остов — 4 ребра по 7, то есть 28, и он корректен


def test_edge_cases():      # граничные случаи
    assert kruskal_mst(build_graph([])) == ([], 0)  # пустой граф: алгоритм не падает и возвращает пустой остов
    assert kruskal_mst(build_graph([], vertices=['x'])) == ([], 0)  # одна вершина: дерево без рёбер
    g = build_graph([(1, 1, 0), (1, 2, 5)])  # петля 1–1 с весом 0 — самое лёгкое ребро графа
    assert kruskal_mst(g) == ([(1, 2, 5)], 5)  # но петля всегда цикл, поэтому в остове только 1–2
    g = build_graph([(1, 2, 9), (1, 2, 3), (2, 3, 4)])  # два ребра между 1 и 2 (кратные рёбра)
    assert kruskal_mst(g)[1] == 7     # взято лёгкое 3, тяжёлое 9 отброшено как цикл: 3 + 4 = 7
    g = build_graph([(1, 2, -5), (2, 3, 2), (1, 3, -1)])  # отрицательные веса — Дейкстру они ломают
    assert kruskal_mst(g)[1] == -6    # а Краскалу всё равно: он только сравнивает веса, берутся -5 и -1
    g = build_graph([('a', 'b', 0.5), ('b', 'c', 1.25), ('a', 'c', 2.0)])  # дробные веса, точно представимые в двоичной записи
    assert kruskal_mst(g)[1] == 1.75  # поэтому сравнение через == не подведёт из-за округления


def test_checker_catches_bad_trees(): # проверяем, что сама функция проверки ловит неправильные остовы
    g = build_graph(WEDGES)
    cycle = [('A', 'B', 4), ('B', 'C', 1), ('A', 'C', 2),  # пять рёбер, но A–B–C — цикл,
             ('D', 'E', 2), ('E', 'F', 3)]                 # и куски {A, B, C} и {D, E, F} между собой не соединены
    assert not is_spanning_forest(g, cycle)  # проверка обязана это заметить
    assert not is_spanning_forest(g, kruskal_mst(g)[0][:-1])  # настоящий остов без последнего ребра — граф распался
    assert not is_spanning_forest(g, [('A', 'F', 1)])  # ребра A–F в графе нет — остов выдуман


def test_random_vs_bruteforce(): # 200 случайных графов: Краскал сверяется с полным перебором и с Примом
    rng = random.Random(20)     # фиксированное зерно: упадёт тест — сможем получить тот же граф снова
    for _ in range(200):    # проверяем 200 разных графов
        n = rng.randint(1, 7)   # до 7 вершин — больше полный перебор считал бы слишком долго
        edges = [(i, rng.randrange(i), rng.randint(1, 5)) for i in range(1, n)]  # каждую вершину цепляем к одной из предыдущих — получается дерево, граф точно связный
        for _ in range(rng.randint(0, 8)):  # добавляем до 8 лишних рёбер, чтобы появились циклы и у алгоритма был выбор
            a, b = rng.sample(range(n), 2) if n > 1 else (0, 0)  # две разные случайные вершины; при n = 1 выбрать две нельзя
            if a != b:                # петли не добавляем
                edges.append((a, b, rng.randint(1, 5)))  # веса от 1 до 5 — много совпадений, заодно проверяем равные веса
        g = build_graph(edges, vertices=range(n))  # vertices нужен для n = 1, когда рёбер нет, а вершина 0 должна быть
        tree, total = kruskal_mst(g)
        assert is_spanning_forest(g, tree)  # структура остова правильная
        assert total == brute_force_mst_weight(g) == prim_weight(g, 0)  # и вес минимальный: три разных способа дали одно число


def run_all():      # запускает все тесты по очереди
    tests = [test_dsu, test_basic, test_edge_sorting, test_disconnected,  # функции в Python — обычные объекты,
             test_equal_weights, test_edge_cases,                         # их можно сложить в список
             test_checker_catches_bad_trees, test_random_vs_bruteforce]
    for t in tests:  # идём по списку тестов
        t()      # вызываем тест; упавший assert остановит прогон и покажет строку с ошибкой
        print('  [ok]', t.__name__)   # дошли сюда — тест прошёл; __name__ — имя функции строкой
    print('Все тесты пройдены:', len(tests))  # итог, если ничего не упало


if __name__ == '__main__':     # истинно только при запуске файла напрямую, а не при импорте —
    run_all()   # поэтому тесты запускаются только когда мы сами запускаем файл